In [1]:
import json

reverse_code_columns = [
    11, 16, 26, 31, 36, 51, 12, 17, 22, 37, 42, 47,
    3, 8, 23, 28, 48, 58, 4, 9, 24, 29, 44, 49,
    5, 25, 30, 45, 50, 55
]

reverse_items = [f"Item{i}" for i in reverse_code_columns]

with open("../PSI_dataset_scored.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for entry in data:
    for item in reverse_items:
        if item in entry:
            entry[item] = 6 - entry[item]

with open("PSI_dataset_scored_unreversed.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)



In [3]:
import os
import json
import numpy as np
import pandas as pd

folder_path = '../All_Data_json'

all_items = [f"Item{i}" for i in range(1, 61)]
dimensions = [
    "Sociability", "Assertiveness", "Energy_Level", "Compassion", "Respectfulness", "Trust",
    "Organization", "Productiveness", "Responsibility", "Anxiety", "Depression",
    "Emotional_Volatility", "Intellectual_Curiosity", "Aesthetic_Sensitivity",
    "Creative_Imagination", "Extraversion", "Agreeableness", "Conscientiousness",
    "Neuroticism", "Openness"
]
all_fields = all_items + dimensions

reverse_code_columns = [
    11, 16, 26, 31, 36, 51, 12, 17, 22, 37, 42, 47,
    3, 8, 23, 28, 48, 58, 4, 9, 24, 29, 44, 49,
    5, 25, 30, 45, 50, 55
]
reverse_cols = [f"Item{i}" for i in reverse_code_columns]

final_df = pd.DataFrame(index=[field.replace("_", " ") for field in all_fields])

for filename in os.listdir(folder_path):
    if filename.endswith(".json"):
        filepath = os.path.join(folder_path, filename)
        print(f"Processing: {filename}")
        
        try:
            with open(filepath, "r", encoding="utf-8") as f:
                records = json.load(f)
        except json.JSONDecodeError as e:
            print(f"Error reading {filename}: {e}")
            continue

        file_data = {field: [] for field in all_fields}
        for record in records:
            for field in all_fields:
                if field in record:
                    value = record[field]
                    if field in reverse_cols and isinstance(value, (int, float)):
                        value = 6 - value
                    file_data[field].append(value)

        file_result = {}
        # for field in all_fields:
        #     values = np.array(file_data[field])
        #     mean = np.mean(values) if len(values) > 0 else np.nan
        #     std = np.std(values, ddof=1) if len(values) > 1 else np.nan
        #     field_name = field.replace("_", " ")
        #     file_result[field_name] = (round(mean, 2), round(std, 2))
        
        for field in all_fields:
            values = np.array(file_data[field], dtype=np.float64)
            values = values[~np.isnan(values)] 
            mean = np.mean(values) if len(values) > 0 else np.nan
            std = np.std(values, ddof=1) if len(values) > 1 else np.nan
            field_name = field.replace("_", " ")
            file_result[field_name] = (round(mean, 2), round(std, 2))

        mean_col = f"{filename}_mean"
        sd_col = f"{filename}_sd"
        file_df = pd.DataFrame({
            mean_col: {k: v[0] for k, v in file_result.items()},
            sd_col: {k: v[1] for k, v in file_result.items()}
        })
        final_df = final_df.join(file_df)

output_file = "Output/all_files_stats_by_column.csv"
final_df.to_csv(output_file)
print(f"\n{output_file}")


Processing: human_normal.json
Processing: persona_Gemma2_27b_zero_shot.json
Processing: persona_Gemma2_9b_zero_shot.json
Processing: persona_GPT4o_mini_zero_shot.json
Processing: persona_GPT4o_zero_shot.json
Processing: persona_Llama3_70b_zero_shot.json
Processing: persona_Llama3_8b_zero_shot.json
Processing: persona_Mistral_7b_zero_shot.json
Processing: PSI_dataset_scored.json
Processing: PSI_dataset_scored_unreversed.json
Processing: psi_Gemma2_27b_zero_shot.json
Processing: psi_Gemma2_9b_zero_shot.json
Processing: psi_GPT4o_mini_zero_shot.json
Processing: psi_GPT4o_zero_shot.json
Processing: psi_Llama3_70b_zero_shot.json
Processing: psi_Llama3_8b_zero_shot.json
Processing: psi_Mistral_7b_zero_shot.json
Processing: shape_Gemma2_27b_zero_shot.json
Processing: shape_Gemma2_9b_zero_shot.json
Processing: shape_GPT4o_mini_zero_shot.json
Processing: shape_GPT4o_zero_shot.json
Processing: shape_Llama3_70b_zero_shot.json
Processing: shape_Llama3_8b_zero_shot.json
Processing: shape_Mistral_7b

In [22]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

# all_df = pd.read_csv("existing.csv", index_col=0)
all_df = pd.read_csv("./Output/all_files_stats_by_column.csv", index_col=0)
human_df = pd.read_csv("human_stats_by_column.csv", index_col=0)

item_group = [f"Item{i}" for i in range(1, 61)]
facet_group = [
    "Sociability", "Assertiveness", "Energy Level", "Compassion", "Respectfulness",
    "Trust", "Organization", "Productiveness", "Responsibility", "Anxiety", "Depression",
    "Emotional Volatility", "Intellectual Curiosity", "Aesthetic Sensitivity", "Creative Imagination"
]
domain_group = ["Extraversion", "Agreeableness", "Conscientiousness", "Neuroticism", "Openness"]

group_dict = {
    "Item": item_group,
    "Facet": facet_group,
    "Domain": domain_group
}

file_names = sorted(set(col.rsplit('_', 1)[0] for col in all_df.columns if not col.startswith("normal_human")))

results = []

for file in file_names:
    mean_col = f"{file}_mean"
    sd_col = f"{file}_sd"

    for group_name, items in group_dict.items():
        model_mean = all_df.loc[items, mean_col]
        model_sd = all_df.loc[items, sd_col]

        human_mean = human_df.loc[items, "normal_human.mean"]
        human_sd = human_df.loc[items, "normal_human.sd"]

        mean_corr, _ = pearsonr(model_mean, human_mean)
        mean_mae = mean_absolute_error(human_mean, model_mean)

        sd_corr, _ = pearsonr(model_sd, human_sd)
        sd_mae = mean_absolute_error(human_sd, model_sd)

        results.append({
            "file": file,
            "group": group_name,
            "mean_corr": mean_corr,
            "mean_mae": mean_mae,
            "sd_corr": sd_corr,
            "sd_mae": sd_mae
        })

result_df = pd.DataFrame(results)

pivot_mae = result_df.pivot(index="file", columns="group", values=["mean_mae", "sd_mae"])
pivot_corr = result_df.pivot(index="file", columns="group", values=["mean_corr", "sd_corr"])

ordered_groups = ["Item", "Facet", "Domain"]

columns = []
metrics = [("mean_mae", "MAE M"), ("mean_corr", "r M"), ("sd_mae", "MAE SD"), ("sd_corr", "r SD")]

for group in ordered_groups:
    for metric_key, metric_label in metrics:
        columns.append(((metric_key, group), (group, metric_label)))

final_df = pd.DataFrame(index=pivot_mae.index)

for (metric_key, group), (top_label, sub_label) in columns:
    if "mae" in metric_key:
        source_df = pivot_mae
    else:
        source_df = pivot_corr
    final_df[(top_label, sub_label)] = source_df[(metric_key, group)]

final_df.columns = pd.MultiIndex.from_tuples(final_df.columns)

final_df.to_csv("./Output/final_grouped_comparison.csv")
print(final_df)


                                        Item                                \
                                       MAE M       r M    MAE SD      r SD   
file                                                                         
PSI_dataset_scored_unreversed.json  0.127833  0.981376  0.118500  0.957434   
human_normal.json                   0.000000  1.000000  0.000000  1.000000   
persona_GPT4o_mini_zero_shot.json   0.332500  0.797251  0.436000  0.002278   
persona_GPT4o_zero_shot.json        0.484500  0.499978  0.435667 -0.130557   
persona_Gemma2_27b_zero_shot.json   0.419333  0.676615  0.594167 -0.235527   
persona_Gemma2_9b_zero_shot.json    0.512333  0.635215  0.616500 -0.470532   
persona_Llama3_70b_zero_shot.json   0.508667  0.574351  0.260000  0.021988   
persona_Llama3_8b_zero_shot.json    0.775833  0.065255  0.448500  0.046094   
persona_Mistral_7b_zero_shot.json   0.423000  0.683759  0.463000 -0.095864   
psi_GPT4o_cot.json                  0.406667  0.729223  0.312333

In [32]:
# TFM model fit

import pandas as pd

# reading the model fit results
df_fit = pd.read_csv('Output/TFM_factor_analysis_model_fit_results.csv', encoding='utf-8-sig')

# mapping
model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

# output order
ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

# mapping trait names to model codes
trait_name_mapping = {
    'ext': 'Extraversion_TFM',
    'agr': 'Agreeableness_TFM',
    'con': 'Conscientiousness_TFM',
    'neu': 'Neuroticism_TFM',
    'ope': 'Openness_TFM'
}

# reverse mapping for model codes to labels
reverse_model_mapping = {v: k for k, v in model_mapping.items()}

# each trait has a corresponding model code
fit_tables = {}

# going through each trait and creating a DataFrame for the fit metrics
for code, trait_name in trait_name_mapping.items():
    df_out = pd.DataFrame(columns=['Chi-square', 'df', 'CFI', 'TLI', 'RMSEA', 'SRMR'])

    for model_label in ordered_models:
        file_code = reverse_model_mapping.get(model_label)
        if file_code is None:
            continue

        row = df_fit[(df_fit['file'] == file_code) & (df_fit['model'] == code)]
        if not row.empty:
            row = row.iloc[0]
            df_out.loc[model_label] = [
                row['chisq'],
                row['df'],
                row['cfi'],
                row['tli'],
                row['rmsea'],
                row['srmr_bentler_nomean']
            ]
    fit_tables[trait_name] = df_out

# saving the fit tables to an Excel file
with pd.ExcelWriter('./Results/TFM_model_fit_metrics.xlsx', engine='openpyxl') as writer:
    for trait_name, df in fit_tables.items():
        df.to_excel(writer, sheet_name=trait_name)



In [39]:
# FFM model fit
import pandas as pd

# read the model fit results
df_ffm_fit = pd.read_csv('Output/FFM_factor_analysis_model_fit_results.csv', encoding='utf-8-sig')

# mapping 
model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

# output order
ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

# reverse mapping for model codes to labels
reverse_model_mapping = {v: k for k, v in model_mapping.items()}

# creating a DataFrame for the fit metrics
output_df = pd.DataFrame(columns=['Chi-square', 'df', 'CFI', 'TLI', 'RMSEA', 'SRMR'])

for model_label in ordered_models:
    file_code = reverse_model_mapping.get(model_label)
    if file_code is None:
        continue

    row = df_ffm_fit[(df_ffm_fit['file'] == file_code) & (df_ffm_fit['model'] == 'FFM')]
    if not row.empty:
        row = row.iloc[0]
        output_df.loc[model_label] = [
            row['chisq'],
            row['df'],
            row['cfi'],
            row['tli'],
            row['rmsea'],
            row['srmr_bentler_nomean']
        ]

# saving the fit metrics to an Excel file
output_df.to_excel('./Results/FFM_model_fit_metrics.xlsx', sheet_name='FFM_Fit', engine='openpyxl')



In [33]:
# TFM factor loadings

import pandas as pd

df_ffm = pd.read_csv('Output/TFM_factor_analysis_factor_loadings.csv', encoding='utf-8-sig')

# mapping 
model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

# output order
ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

col_mapping = {
    'ext': { # Extraversion_TFM
        'item1': 'Sociability_item1',
        'item16': 'Sociability_item16',
        'item31': 'Sociability_item31',
        'item46': 'Sociability_item46',
        'item6': 'Assertiveness_item6',
        'item21': 'Assertiveness_item21',
        'item36': 'Assertiveness_item36',
        'item51': 'Assertiveness_item51',
        'item11': 'Energy_Level_item11',
        'item26': 'Energy_Level_item26',
        'item41': 'Energy_Level_item41',
        'item56': 'Energy_Level_item56'
    },
    'agr': { # Agreeableness_TFM
        'item2': 'Compassion_item2',
        'item17': 'Compassion_item17',
        'item32': 'Compassion_item32',
        'item47': 'Compassion_item47',
        'item7': 'Respectfulness_item7',
        'item22': 'Respectfulness_item22',
        'item37': 'Respectfulness_item37',
        'item52': 'Respectfulness_item52',
        'item12': 'Trust_item12',
        'item27': 'Trust_item27',
        'item42': 'Trust_item42',
        'item57': 'Trust_item57'
    },
    'con': { # Conscientiousness_TFM
        'item3': 'Organization_item3',
        'item18': 'Organization_item18',
        'item33': 'Organization_item33',
        'item48': 'Organization_item48',
        'item8': 'Productiveness_item8',
        'item23': 'Productiveness_item23',
        'item38': 'Productiveness_item38',
        'item53': 'Productiveness_item53',
        'item13': 'Responsibility_item13',
        'item28': 'Responsibility_item28',
        'item43': 'Responsibility_item43',
        'item58': 'Responsibility_item58'
    },
    'neu': { # Neuroticism_TFM
        'item4': 'Anxiety_item4',
        'item19': 'Anxiety_item19',
        'item34': 'Anxiety_item34',
        'item49': 'Anxiety_item49',
        'item9': 'Depression_item9',
        'item24': 'Depression_item24',
        'item39': 'Depression_item39',
        'item54': 'Depression_item54',
        'item14': 'Emotional_Volatility_item14',
        'item29': 'Emotional_Volatility_item29',
        'item44': 'Emotional_Volatility_item44',
        'item59': 'Emotional_Volatility_item59'
    },
    'ope': { # Openness_TFM
        'item10': 'Intellectual_Curiosity_item10',
        'item25': 'Intellectual_Curiosity_item25',
        'item40': 'Intellectual_Curiosity_item40',
        'item55': 'Intellectual_Curiosity_item55',
        'item5': 'Aesthetic_Sensitivity_item5',
        'item20': 'Aesthetic_Sensitivity_item20',
        'item35': 'Aesthetic_Sensitivity_item35',
        'item50': 'Aesthetic_Sensitivity_item50',
        'item15': 'Creative_Imagination_item15',
        'item30': 'Creative_Imagination_item30',
        'item45': 'Creative_Imagination_item45',
        'item60': 'Creative_Imagination_item60'
    }
}

# all traits
trait_tables = {}

# loop through each trait and create a DataFrame for each
for trait, item_map in col_mapping.items():
    trait_name = {
        'ext': 'Extraversion_TFM',
        'agr': 'Agreeableness_TFM',
        'con': 'Conscientiousness_TFM',
        'neu': 'Neuroticism_TFM',
        'ope': 'Openness_TFM'
    }[trait]
    
    columns = [item_map[i] for i in item_map]
    output_df = pd.DataFrame(columns=columns)

    # Create a new index for the DataFrame
    reverse_model_mapping = {v: k for k, v in model_mapping.items()}

    for model_label in ordered_models:
        model_code = reverse_model_mapping.get(model_label)
        if model_code is None:
            continue

        rows = df_ffm[(df_ffm['file'] == model_code) & (df_ffm['model'] == trait)]
        row_data = {}
        for _, row in rows.iterrows():
            item = row['item']
            if item in item_map:
                new_col = item_map[item]
                row_data[new_col] = row['loading']
        output_df.loc[model_label] = row_data

    trait_tables[trait_name] = output_df

# Save each trait DataFrame to a separate sheet in an Excel file
with pd.ExcelWriter('./Results/TFM_all_traits_ordered.xlsx', engine='openpyxl') as writer:
    for trait_name, df in trait_tables.items():
        df.to_excel(writer, sheet_name=trait_name)


In [34]:
# FFM factor loadings

import pandas as pd

# read the FFM factor loadings CSV file
df_ffm = pd.read_csv('Output/FFM_factor_analysis_factor_loadings.csv', encoding='utf-8-sig')

# mapping
model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

# output order
ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

# facet grouping
ffm_grouping = {
    'Extraversion_FFM': ['Sociability', 'Assertiveness', 'Energy_Level'],
    'Agreeableness_FFM': ['Compassion', 'Respectfulness', 'Trust'],
    'Conscientiousness_FFM': ['Organization', 'Productiveness', 'Responsibility'],
    'Neuroticism_FFM': ['Anxiety', 'Depression', 'Emotional_Volatility'],
    'Openness_FFM': ['Intellectual_Curiosity', 'Aesthetic_Sensitivity', 'Creative_Imagination']
}

# output dictionary
ffm_tables = {}

# reverse mapping for model names
reverse_model_mapping = {v: k for k, v in model_mapping.items()}

# loop through each FFM trait and create a DataFrame for each
for ffm_trait, subtraits in ffm_grouping.items():
    output_df = pd.DataFrame(columns=subtraits)
    
    for model_label in ordered_models:
        model_code = reverse_model_mapping.get(model_label)
        if model_code is None:
            continue
        rows = df_ffm[(df_ffm['file'] == model_code) & (df_ffm['model'] == 'FFM')]
        
        row_data = {}
        for subtrait in subtraits:
            match = rows[rows['item'] == subtrait]
            if not match.empty:
                row_data[subtrait] = match['loading'].values[0]
        output_df.loc[model_label] = row_data

    ffm_tables[ffm_trait] = output_df

# save each FFM trait DataFrame to a separate sheet in an Excel file
with pd.ExcelWriter('./Results/FFM_all_traits_ordered.xlsx', engine='openpyxl') as writer:
    for trait_name, df in ffm_tables.items():
        df.to_excel(writer, sheet_name=trait_name)


In [35]:
# TFM factor correlations

import pandas as pd

# read the TFM factor correlations CSV file
df_corr = pd.read_csv('Output/TFM_factor_analysis_factor_correlations.csv', encoding='utf-8-sig')

# mapping
model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

# output order
ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

# domain mapping
trait_code_mapping = {
    'Extraversion_TFM': 'ext',
    'Agreeableness_TFM': 'agr',
    'Conscientiousness_TFM': 'con',
    'Neuroticism_TFM': 'neu',
    'Openness_TFM': 'ope'
}

# each trait's factor pairs
tfm_factor_pairs = {
    'Extraversion_TFM': [
        ('Sociability', 'Assertiveness'),
        ('Sociability', 'Energy_Level'),
        ('Assertiveness', 'Energy_Level')
    ],
    'Agreeableness_TFM': [
        ('Compassion', 'Respectfulness'),
        ('Compassion', 'Trust'),
        ('Respectfulness', 'Trust')
    ],
    'Conscientiousness_TFM': [
        ('Organization', 'Productiveness'),
        ('Organization', 'Responsibility'),
        ('Productiveness', 'Responsibility')
    ],
    'Neuroticism_TFM': [
        ('Anxiety', 'Depression'),
        ('Anxiety', 'Emotional_Volatility'),
        ('Depression', 'Emotional_Volatility')
    ],
    'Openness_TFM': [
        ('Intellectual_Curiosity', 'Aesthetic_Sensitivity'),
        ('Intellectual_Curiosity', 'Creative_Imagination'),
        ('Aesthetic_Sensitivity', 'Creative_Imagination')
    ]
}

# reverse mapping for model names
reverse_model_mapping = {v: k for k, v in model_mapping.items()}

# output dictionary
correlation_tables = {}

for trait_name, factor_pairs in tfm_factor_pairs.items():
    model_code = trait_code_mapping[trait_name]
    col_names = [f"{a}~{b}" for a, b in factor_pairs]
    df_out = pd.DataFrame(columns=col_names)

    for model_label in ordered_models:
        file_code = reverse_model_mapping.get(model_label)
        if file_code is None:
            continue

        row_data = {}
        for a, b in factor_pairs:
            match = df_corr[
                (df_corr['file'] == file_code) &
                (df_corr['model'] == model_code) &
                (
                    ((df_corr['factor1'] == a) & (df_corr['factor2'] == b)) |
                    ((df_corr['factor1'] == b) & (df_corr['factor2'] == a))
                )
            ]
            if not match.empty:
                row_data[f"{a}~{b}"] = match['correlation'].values[0]

        df_out.loc[model_label] = row_data

    correlation_tables[trait_name] = df_out

# save each trait DataFrame to a separate sheet in an Excel file
with pd.ExcelWriter('./Results/TFM_factor_correlations_ordered.xlsx', engine='openpyxl') as writer:
    for trait_name, df in correlation_tables.items():
        df.to_excel(writer, sheet_name=trait_name)





In [36]:
# FFM factor correlations

import pandas as pd

# read the FFM factor correlations CSV file
df_ffm_corr = pd.read_csv('Output/FFM_factor_analysis_factor_correlations.csv', encoding='utf-8-sig')

# mapping
model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

# output order
ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

# FMM factor pairs
ffm_pairs = [
    ('Extraversion', 'Agreeableness'),
    ('Extraversion', 'Conscientiousness'),
    ('Extraversion', 'Neuroticism'),
    ('Extraversion', 'Openness'),
    ('Agreeableness', 'Conscientiousness'),
    ('Agreeableness', 'Neuroticism'),
    ('Agreeableness', 'Openness'),
    ('Conscientiousness', 'Neuroticism'),
    ('Conscientiousness', 'Openness'),
    ('Neuroticism', 'Openness'),
]

# column names for the output DataFrame
col_names = [f"{a}~~{b}" for a, b in ffm_pairs]
ffm_corr_df = pd.DataFrame(columns=col_names)

# reverse mapping for model names
reverse_model_mapping = {v: k for k, v in model_mapping.items()}

# entering data into the DataFrame
for model_label in ordered_models:
    file_code = reverse_model_mapping.get(model_label)
    if file_code is None:
        continue

    row_data = {}
    for a, b in ffm_pairs:
        match = df_ffm_corr[
            (df_ffm_corr['file'] == file_code) &
            (df_ffm_corr['model'] == 'FFM') &
            (
                ((df_ffm_corr['factor1'] == a) & (df_ffm_corr['factor2'] == b)) |
                ((df_ffm_corr['factor1'] == b) & (df_ffm_corr['factor2'] == a))
            )
        ]
        if not match.empty:
            row_data[f"{a}~~{b}"] = match['correlation'].values[0]

    ffm_corr_df.loc[model_label] = row_data

# save the DataFrame to an Excel file
ffm_corr_df.to_excel('./Results/FFM_factor_correlations_matrix.xlsx', sheet_name='FFM_Correlations', engine='openpyxl')

In [37]:
# cronbach alpha

import pandas as pd

# read the cronbach alpha results CSV file
df_alpha = pd.read_csv('Output/cronbach_alpha_results.csv', encoding='utf-8-sig')

# mapping
model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

# output order
ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

# same for facet + domain
df_alpha['scale'] = df_alpha['scale'].str.replace(" ", "_")

# create a mapping for the scale to domain/facet
df_alpha['model_pretty'] = df_alpha['file'].map(model_mapping)
pivot = df_alpha.pivot(index='model_pretty', columns='scale', values='alpha')

# order the columns
column_order = [
    # Facets
    "Sociability", "Assertiveness", "Energy_Level",
    "Compassion", "Respectfulness", "Trust",
    "Organization", "Productiveness", "Responsibility",
    "Anxiety", "Depression", "Emotional_Volatility",
    "Intellectual_Curiosity", "Aesthetic_Sensitivity", "Creative_Imagination",
    # Domains
    "Extraversion", "Agreeableness", "Conscientiousness", "Neuroticism", "Openness"
]

columns_in_data = [col for col in column_order if col in pivot.columns]
pivot = pivot[columns_in_data]
pivot = pivot.reindex(ordered_models)

# save the DataFrame to an Excel file
pivot.to_excel('./Results/cronbach_alpha_matrix.xlsx', sheet_name='Alpha', engine='openpyxl')


In [3]:
import os
import json
import pandas as pd
import numpy as np
from itertools import combinations
from scipy.stats import pearsonr

# directory containing the JSON files
data_dir = "../All_Data_json"

# mapping of model names to pretty names
model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

# output order
ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

traits = ["Extraversion", "Agreeableness", "Conscientiousness", "Neuroticism", "Openness"]
pair_labels = ["E-A", "E-C", "E-N", "E-O", "A-C", "A-N", "A-O", "C-N", "C-O", "N-O"]
trait_pairs = list(combinations(traits, 2))

results = {}
for filename in os.listdir(data_dir):
    if filename.endswith(".json"):
        file_path = os.path.join(data_dir, filename)
        with open(file_path, "r") as f:
            data = json.load(f)

        model_key = filename.replace(".json", "")
        model_name = model_mapping.get(model_key, model_key)

        trait_data = {trait: [] for trait in traits}
        for entry in data:
            for trait in traits:
                trait_data[trait].append(entry[trait])

        correlations = []
        for t1, t2 in trait_pairs:
            r, _ = pearsonr(trait_data[t1], trait_data[t2])
            correlations.append(r)

        mean_abs = np.mean(np.abs(correlations))
        results[model_name] = correlations + [mean_abs]

df = pd.DataFrame.from_dict(results, orient="index", columns=pair_labels + ["Mean of absolute values"])
df = df.loc[ordered_models] 

# save the DataFrame to a CSV file
df.to_csv("./Results/trait_correlations_summary.csv", index=True)
print(df)


                                  E-A       E-C       E-N       E-O       A-C  \
Mistral-7B-Instruct+Persona  0.423176  0.547454 -0.491998  0.295532  0.711858   
Mistral-7B-Instruct+Shape    0.480511  0.585434 -0.555865  0.633084  0.750114   
Mistral-7B-Instruct+PSI      0.454774  0.459490 -0.448705  0.433735  0.638465   
Gemma-2-9B-IT+Persona        0.228348  0.430467 -0.456341  0.379794  0.619378   
Gemma-2-9B-IT+Shape          0.572722  0.540124 -0.536935  0.782822  0.669547   
Gemma-2-9B-IT+PSI            0.371022  0.504533 -0.492577  0.464078  0.571858   
Gemma-2-27B-IT+Persona       0.408785  0.539068 -0.556188  0.435177  0.700504   
Gemma-2-27B-IT+Shape         0.572332  0.556534 -0.619729  0.777220  0.783377   
Gemma-2-27B-IT+PSI           0.316779  0.356256 -0.306822  0.343854  0.568120   
Llama3-8B-Instruct+Persona   0.240473  0.263601 -0.386235  0.259017  0.452529   
Llama3-8B-Instruct+Shape     0.281405  0.242471 -0.390290  0.594640  0.551853   
Llama3-8B-Instruct+PSI      

In [43]:
# TFM TCC and MAE

import pandas as pd

df_tcc_mae = pd.read_csv('Output/TFM_TCC_MAE_results.csv', encoding='utf-8-sig')

model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

df_tcc_mae['model_pretty'] = df_tcc_mae['file'].map(model_mapping)

df_tcc_mae['factor'] = df_tcc_mae['factor'].str.replace(" ", "_")

tcc_df = df_tcc_mae.pivot(index='model_pretty', columns='factor', values='tcc').reindex(ordered_models)
mae_df = df_tcc_mae.pivot(index='model_pretty', columns='factor', values='mae').reindex(ordered_models)

column_order = [
    "Sociability", "Assertiveness", "Energy_Level",
    "Compassion", "Respectfulness", "Trust",
    "Organization", "Productiveness", "Responsibility",
    "Anxiety", "Depression", "Emotional_Volatility",
    "Intellectual_Curiosity", "Aesthetic_Sensitivity", "Creative_Imagination"
]

tcc_df = tcc_df[[c for c in column_order if c in tcc_df.columns]]
mae_df = mae_df[[c for c in column_order if c in mae_df.columns]]

with pd.ExcelWriter('./Results/TFM_TCC_MAE_matrix.xlsx', engine='openpyxl') as writer:
    tcc_df.to_excel(writer, sheet_name='TCC')
    mae_df.to_excel(writer, sheet_name='MAE')


In [ ]:
# FFM TCC and MAE

import pandas as pd

df_ffm_tcc_mae = pd.read_csv('Output/FFM_TCC_MAE_results.csv', encoding='utf-8-sig')

model_mapping = {
    "human_normal": "Human sample",
    "persona_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Persona",
    "persona_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Persona",
    "persona_GPT4o_mini_zero_shot": "GPT4o_mini+Persona",
    "persona_GPT4o_zero_shot": "GPT4o+Persona",
    "persona_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Persona",
    "persona_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Persona",
    "persona_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Persona",
    "psi_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+PSI",
    "psi_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+PSI",
    "psi_GPT4o_mini_zero_shot": "GPT4o_mini+PSI",
    "psi_GPT4o_zero_shot": "GPT4o+PSI",
    "psi_Llama3_8b_zero_shot": "Llama3-8B-Instruct+PSI",
    "psi_Llama3_70b_zero_shot": "Llama3-70B-Instruct+PSI",
    "psi_Mistral_7b_zero_shot": "Mistral-7B-Instruct+PSI",
    "shape_Gemma2_9b_zero_shot": "Gemma-2-9B-IT+Shape",
    "shape_Gemma2_27b_zero_shot": "Gemma-2-27B-IT+Shape",
    "shape_GPT4o_mini_zero_shot": "GPT4o_mini+Shape",
    "shape_GPT4o_zero_shot": "GPT4o+Shape",
    "shape_Llama3_8b_zero_shot": "Llama3-8B-Instruct+Shape",
    "shape_Llama3_70b_zero_shot": "Llama3-70B-Instruct+Shape",
    "shape_Mistral_7b_zero_shot": "Mistral-7B-Instruct+Shape"
}

ordered_models = [
    "Mistral-7B-Instruct+Persona",
    "Mistral-7B-Instruct+Shape",
    "Mistral-7B-Instruct+PSI",
    "Gemma-2-9B-IT+Persona",
    "Gemma-2-9B-IT+Shape",
    "Gemma-2-9B-IT+PSI",
    "Gemma-2-27B-IT+Persona",
    "Gemma-2-27B-IT+Shape",
    "Gemma-2-27B-IT+PSI",
    "Llama3-8B-Instruct+Persona",
    "Llama3-8B-Instruct+Shape",
    "Llama3-8B-Instruct+PSI",
    "Llama3-70B-Instruct+Persona",
    "Llama3-70B-Instruct+Shape",
    "Llama3-70B-Instruct+PSI",
    "GPT4o_mini+Persona",
    "GPT4o_mini+Shape",
    "GPT4o_mini+PSI",
    "GPT4o+Persona",
    "GPT4o+Shape",
    "GPT4o+PSI",
    "Human sample"
]

df_ffm_tcc_mae['model_pretty'] = df_ffm_tcc_mae['file'].map(model_mapping)

tcc_df = df_ffm_tcc_mae.pivot(index='model_pretty', columns='factor', values='tcc').reindex(ordered_models)
mae_df = df_ffm_tcc_mae.pivot(index='model_pretty', columns='factor', values='mae').reindex(ordered_models)

column_order = ['Extraversion', 'Agreeableness', 'Conscientiousness', 'Neuroticism', 'Openness']
tcc_df = tcc_df[[c for c in column_order if c in tcc_df.columns]]
mae_df = mae_df[[c for c in column_order if c in mae_df.columns]]

with pd.ExcelWriter('./Results/FFM_TCC_MAE_matrix.xlsx', engine='openpyxl') as writer:
    tcc_df.to_excel(writer, sheet_name='TCC')
    mae_df.to_excel(writer, sheet_name='MAE')
